# DCJ quarterly sales workbooks

This notebook inspects issues 128–156, covering March 2019 through March 2026. The NSW Department of Communities and Justice (DCJ) published all 29 quarters. Its Greater Sydney grouping contains 34 LGAs across three rings. Wollondilly has no published strata median in 2019. The June issue omits the row, and the other three issues report a dash.

Start Jupyter from the repository root or run the nbconvert command in T01. The notebook reads the original workbooks under `data/raw/dcj_sales/`. Quarters use the first day of the quarter when shown as a date. [DCJ sales explanatory notes](https://dcj.nsw.gov.au/about-us/families-and-communities-statistics/housing-rent-and-sales/explanatory-notes.html)

## 1. Files

List the workbooks, their byte sizes and sheet names. Read the values from the `LGA` sheet and keep their Excel row positions so each observation can be traced to the source.

In [1]:
from pathlib import Path
import re
from string import ascii_uppercase

import pandas as pd
from IPython.display import display
from openpyxl import load_workbook

raw_dir = Path("data/raw")
if Path.cwd().name == "notebooks":
    raw_dir = Path("../data/raw")
sales_dir = raw_dir / "dcj_sales"

files = []
lga_sheets = {}
for path in sorted(sales_dir.glob("*.xlsx")):
    workbook = load_workbook(path, read_only=True, data_only=True)
    try:
        files.append({
            "filename": path.name,
            "bytes": path.stat().st_size,
            "sheets": ", ".join(workbook.sheetnames),
        })
        lga_sheets[path.name] = list(workbook["LGA"].values)
    finally:
        workbook.close()

file_inventory = pd.DataFrame(files).set_index("filename")
display(file_inventory)

,bytes,sheets
filename,,
Sales_tables_September_2024_quarter.xlsx,471111,"Explanatory Notes, LGA, Postcode"
issue-128-sales-tables-march-2019.xlsx,380498,"Explanatory Notes, LGA, Postcode"
issue-129-sales-tables-june-2019.xlsx,550885,"Explanatory Notes, LGA, Postcode"
issue-130-sales-tables-september-2019.xlsx,389253,"Explanatory Notes, LGA, Postcode"
issue-131-sales-tables-december-2019.xlsx,393825,"Explanatory Notes, LGA, Postcode"
issue-132-sales-tables-march-2020.xlsx,631019,"Explanatory Notes, LGA, Postcode"
issue-133-sales-tables-june-2020.xlsx,595512,"Explanatory Notes, LGA, Postcode"
issue-134-sales-tables-september-2020.xlsx,604710,"Explanatory Notes, LGA, Postcode"
issue-135-sales-tables-december-2020.xlsx,610082,"Explanatory Notes, LGA, Postcode"


## 2. Layout

Inspect the latest workbook’s opening rows and the exact Excel headers. Its sales table has one reporting quarter. Dwelling type is recorded in column E as `Non Strata`, `Strata` or `Total`. The layout is long by dwelling type and wide by measure, with separate columns for the price statistics, sales count and quarterly and annual change. Prices are in thousands of dollars.

Find the header in every issue because its row number changes. Read the reporting quarter from the title in older files and the period line in newer files.

In [2]:
representative = "sales-tables-march-2026-quarter.xlsx"
columns = [
    "gmr", "greater_sydney", "ring", "lga", "dwelling_type",
    "first_quartile_price", "median_price", "third_quartile_price",
    "mean_price", "sales_count", "quarterly_median_change", "annual_median_change",
    "quarterly_count_change", "annual_count_change",
]
excel_columns = list(ascii_uppercase[:len(columns)])

layout = []
for filename, rows in lga_sheets.items():
    header_index = next(index for index, row in enumerate(rows) if row[4] == "DwellingType")
    period_caption = rows[0][0]
    if rows[1][0].startswith("Reporting period:"):
        period_caption = rows[1][0]
    # Both caption styles end in the quarter-end month and year.
    end_month, year = re.search(
        r"(March|June|September|December)(?: Quarter)? (\d{4})$", period_caption
    ).groups()
    quarter_end = pd.to_datetime(f"{end_month} {year}", format="%B %Y")
    layout.append({
        "filename": filename,
        "quarter": quarter_end.to_period("Q"),
        "quarter_date": quarter_end.to_period("Q").start_time,
        "header_row": header_index + 1,
        "period_caption": period_caption,
    })

layout = pd.DataFrame(layout).sort_values("quarter").set_index("filename")
latest_rows = lga_sheets[representative]
header_row = layout.loc[representative, "header_row"]
for excel_row, row in enumerate(latest_rows[:header_row - 1], start=1):
    if row[0] is not None:
        print(f"A{excel_row}  {row[0]}")

print(f"Header row in {representative} is {header_row}.")
display(pd.DataFrame({
    "excel_column": excel_columns,
    "source_header": [label.replace("\n", " ") for label in latest_rows[header_row - 1]],
    "dataframe_column": columns,
}))
display(layout.groupby("header_row").size().rename("workbooks").to_frame())

A1  Table 3. Sale prices statistics by NSW Local Government Area
A2  Reporting period: January - March 2026
A3  (s): 30 or fewer properties sold; (-): 10 or fewer properties sold
A4  Read explanatory notes
Header row in sales-tables-march-2026-quarter.xlsx is 7.


,excel_column,source_header,dataframe_column
0,A,Greater Metropolitan Region (GMR),gmr
1,B,Greater Sydney,greater_sydney
2,C,Rings,ring
3,D,Local Government Area (LGA),lga
4,E,DwellingType,dwelling_type
5,F,First Quartile Sales Price $'000s,first_quartile_price
6,G,Median Sales Price $'000s,median_price
7,H,Third Quartile Sales Price '000s,third_quartile_price
8,I,Mean Sales Price $'000s,mean_price
9,J,Sales No.,sales_count


,workbooks
header_row,
5,2
6,1
7,26


## 3. Load

Stack the published data rows from each `LGA` sheet. Use column positions because some header wording changes across issues. Keep the original cell values and attach the source filename, Excel row and reporting quarter. The `object` dtype preserves mixed Excel numbers, text and suppression markers.

In [3]:
tables = []
for filename, info in layout.iterrows():
    source_rows = lga_sheets[filename][info["header_row"]:]
    table = pd.DataFrame(source_rows, columns=columns, dtype=object)
    table.insert(0, "excel_row", range(info["header_row"] + 1, info["header_row"] + len(table) + 1))
    table.insert(0, "filename", filename)
    table.insert(0, "quarter_date", info["quarter_date"])
    # The published table has a dwelling type on each data row.
    tables.append(table.loc[table["dwelling_type"].notna()])

raw_sales = pd.concat(tables, ignore_index=True)
print("Raw LGA-sheet shape", raw_sales.shape)
display(raw_sales.dtypes.rename("dtype").to_frame())
# Show the first five rows in two parts so the full source preview fits on screen.
display(raw_sales.head().iloc[:, :10])
display(raw_sales.head().iloc[:, 10:])

Raw LGA-sheet shape (81127, 17)


,dtype
quarter_date,datetime64[us]
filename,str
excel_row,int64
gmr,object
greater_sydney,object
ring,object
lga,object
dwelling_type,object
first_quartile_price,object
median_price,object


,quarter_date,filename,excel_row,gmr,greater_sydney,ring,lga,dwelling_type,first_quartile_price,median_price
0,2019-01-01,issue-128-sales-tables-march-2019.xlsx,6,Total,Total,Total,Total,Total,455,635
1,2019-01-01,issue-128-sales-tables-march-2019.xlsx,7,Total,Total,Total,Total,Non Strata,440,630
2,2019-01-01,issue-128-sales-tables-march-2019.xlsx,8,Total,Total,Total,Total,Strata,485,640
3,2019-01-01,issue-128-sales-tables-march-2019.xlsx,9,Total,Total,Total,Albury,Total,240,320
4,2019-01-01,issue-128-sales-tables-march-2019.xlsx,10,Total,Total,Total,Albury,Non Strata,260,352


,third_quartile_price,mean_price,sales_count,quarterly_median_change,annual_median_change,quarterly_count_change,annual_count_change
0,870,764,19558,-0.0231,-0.0508,-0.2309,-0.3087
1,895,779,13598,-0.0308,-0.0526,-0.233,-0.2873
2,830,730,5960,-0.0303,-0.0448,-0.2262,-0.353
3,415,340,198,0.0492,0.0492,-0.2414,-0.2747
4,438,370,162,-0.0154,0.043,-0.243,-0.2249


## 4. Coverage

Compare the periods found in the workbook titles with the expected quarterly sequence. Count distinct labels and geography combinations. DCJ repeats LGA rows under several levels of its hierarchy. For one statewide view of the LGA rows, use the block where the first three geography columns read `Total`. For the Sydney LGA view, use the rows explicitly assigned to a ring within Greater Sydney and GMR.

In [4]:
expected = pd.period_range("2019Q1", "2026Q1", freq="Q")
observed = pd.PeriodIndex(layout["quarter"].unique()).sort_values()
print("First quarter", observed.min())
print("Latest quarter", observed.max())
print("Distinct quarters", len(observed))
print("Missing quarters", expected.difference(observed).astype(str).tolist())

geography_columns = ["gmr", "greater_sydney", "ring", "lga"]
rings = ["Inner Ring", "Middle Ring", "Outer Ring"]
gmr_name = "GMR (Greater Metropolitan Region)"
lga_rows = raw_sales.loc[raw_sales["lga"].ne("Total")]
state_lgas = lga_rows.loc[
    lga_rows["gmr"].eq("Total")
    & lga_rows["greater_sydney"].eq("Total")
    & lga_rows["ring"].eq("Total")
]
sydney_lgas = lga_rows.loc[
    lga_rows["gmr"].eq(gmr_name)
    & lga_rows["greater_sydney"].eq("Greater Sydney")
    & lga_rows["ring"].isin(rings)
]

print("Distinct A–D geography combinations", len(raw_sales[geography_columns].drop_duplicates()))
print("Distinct LGA labels", state_lgas["lga"].nunique())
print("Distinct Sydney LGA labels", sydney_lgas["lga"].nunique())
coverage = layout[["quarter", "header_row"]].copy()
coverage["lga_sheet_rows"] = raw_sales.groupby("filename").size()
coverage["distinct_lga_labels"] = state_lgas.groupby("filename")["lga"].nunique()
coverage["sydney_lga_labels"] = sydney_lgas.groupby("filename")["lga"].nunique()
display(coverage)

First quarter 2019Q1
Latest quarter 2026Q1
Distinct quarters 29
Missing quarters []


Distinct A–D geography combinations 1057
Distinct LGA labels 128
Distinct Sydney LGA labels 34


,quarter,header_row,lga_sheet_rows,distinct_lga_labels,sydney_lga_labels
filename,,,,,
issue-128-sales-tables-march-2019.xlsx,2019Q1,5,2747,127,34
issue-129-sales-tables-june-2019.xlsx,2019Q2,5,2747,127,34
issue-130-sales-tables-september-2019.xlsx,2019Q3,6,2763,127,34
issue-131-sales-tables-december-2019.xlsx,2019Q4,7,2787,127,34
issue-132-sales-tables-march-2020.xlsx,2020Q1,7,2755,126,34
issue-133-sales-tables-june-2020.xlsx,2020Q2,7,2739,127,34
issue-134-sales-tables-september-2020.xlsx,2020Q3,7,2803,128,34
issue-135-sales-tables-december-2020.xlsx,2020Q4,7,2827,127,34
issue-136-sales-tables-march-2021.xlsx,2021Q1,7,2835,128,34


## 5. Geography labels

List every LGA label and how often it appears, with DCJ’s Sydney rings shown beside the corresponding names. Compare each issue with the union of LGA labels and check whether a Sydney LGA changes rings. List the source’s aggregate geography combinations using rows where column D is `Total`.

In [5]:
sydney_membership = sydney_lgas[["lga", "ring"]].drop_duplicates()
lga_labels = (
    state_lgas.groupby("lga")["quarter_date"].nunique().rename("quarters")
    .to_frame()
    .join(sydney_membership.set_index("lga").rename(columns={"ring": "dcj_sydney_ring"}))
)
with pd.option_context("display.max_rows", None):
    display(lga_labels)

print("LGA labels absent from some quarters")
for quarter, quarter_lgas in state_lgas.groupby("quarter_date"):
    missing_labels = sorted(set(lga_labels.index) - set(quarter_lgas["lga"]))
    if missing_labels:
        print(quarter.to_period("Q"), ", ".join(missing_labels))

print("Sydney ring membership")
display(sydney_membership.groupby("ring")["lga"].nunique().rename("lgas").to_frame())
ring_changes = sydney_membership.groupby("lga")["ring"].nunique().gt(1)
print("Sydney LGAs that appear under multiple rings", ring_changes[ring_changes].index.tolist())

aggregates = raw_sales.loc[raw_sales["lga"].eq("Total"), geography_columns]
aggregate_combinations = aggregates.drop_duplicates().sort_values(geography_columns)
print("Published aggregate combinations", len(aggregate_combinations))
display(aggregate_combinations.reset_index(drop=True))

,quarters,dcj_sydney_ring
lga,,
Albury,29,NaN
Armidale Regional,29,NaN
Ballina,29,NaN
Balranald,27,NaN
Bathurst Regional,29,NaN
Bayside,29,Middle Ring
Bega Valley,29,NaN
Bellingen,29,NaN
Berrigan,29,NaN


LGA labels absent from some quarters
2019Q1 Brewarrina
2019Q2 Brewarrina
2019Q3 Brewarrina
2019Q4 Bourke
2020Q1 Brewarrina, Central Darling
2020Q2 Bogan
2020Q4 Brewarrina
2021Q2 Brewarrina
2021Q3 Brewarrina
2022Q4 Brewarrina
2023Q3 Balranald
2024Q1 Brewarrina
2024Q4 Brewarrina
2025Q2 Brewarrina
2026Q1 Balranald
Sydney ring membership


,lgas
ring,
Inner Ring,8
Middle Ring,12
Outer Ring,14


Sydney LGAs that appear under multiple rings []
Published aggregate combinations 33


,gmr,greater_sydney,ring,lga
0,GMR (Greater Metropolitan Region),Greater Sydney,Inner Ring,Total
1,GMR (Greater Metropolitan Region),Greater Sydney,Middle Ring,Total
2,GMR (Greater Metropolitan Region),Greater Sydney,Outer Ring,Total
3,GMR (Greater Metropolitan Region),Greater Sydney,Total,Total
4,GMR (Greater Metropolitan Region),Rest of GMR,Newcastle SSD,Total
5,GMR (Greater Metropolitan Region),Rest of GMR,Total,Total
6,GMR (Greater Metropolitan Region),Rest of GMR,Wollongong SSD,Total
7,GMR (Greater Metropolitan Region),Total,Inner Ring,Total
8,GMR (Greater Metropolitan Region),Total,Middle Ring,Total
9,GMR (Greater Metropolitan Region),Total,Newcastle SSD,Total


The distinct spellings are consistent across the issues checked. The LGA omissions listed above are changes in row presence. DCJ labels the non-GMR remainder `Rest of State`. Its Inner, Middle and Outer rings include Central Coast, Blue Mountains, Hawkesbury and Wollondilly in the Outer Ring.

## 6. Missing and suppressed values

Count each non-numeric marker in the four price statistics, sales count and change fields. Show the whole published sheet, the statewide LGA block and the DCJ Sydney LGA rows. Numeric strings, commas in counts and percent signs are Excel display formats. The audit recognises them without altering the raw DataFrame.

DCJ says it suppresses results for groups with 10 sales or fewer. The table uses `-` for suppression and `s` in the count field for small samples up to 30. [DCJ sales explanatory notes](https://dcj.nsw.gov.au/about-us/families-and-communities-statistics/housing-rent-and-sales/explanatory-notes.html)

In [6]:
measure_columns = columns[5:]

def as_number(values):
    text = values.astype("string").str.replace(",", "", regex=False)
    return pd.to_numeric(text.str.removesuffix("%"), errors="coerce")

def marker_counts(table):
    counts = {}
    for column in measure_columns:
        values = table[column]
        markers = values.loc[as_number(values).isna()].fillna("<blank>")
        for marker, count in markers.value_counts().items():
            counts[(column, marker)] = count
    return pd.Series(counts, dtype="int64")

markers = pd.concat({
    "all_lga_sheet_rows": marker_counts(raw_sales),
    "statewide_lga_rows": marker_counts(state_lgas),
    "dcj_sydney_lga_rows": marker_counts(sydney_lgas),
}, axis=1).reindex(measure_columns, level=0)
# A zero here is a frequency of a marker, never a replacement for a source value.
markers = markers.fillna(0).astype(int)
markers.index.names = ["measure", "marker"]
display(markers)

all_lga_sheet_rows  statewide_lga_rows  \
measure                 marker                                            
first_quartile_price    -                      8016                1002   
median_price            -                      8016                1002   
third_quartile_price    -                      8016                1002   
mean_price              -                      8016                1002   
sales_count             s                     12992                1624   
                        -                      8016                1002   
quarterly_median_change -                      8016                1002   
                        <blank>                  32                   4   
annual_median_change    -                      8016                1002   
                        <blank>                  32                   4   
quarterly_count_change  -                      8016                1002   
                        <blank>                  32                   4   
annual_count_change     -                      8016                1002   
                        <blank>                  32                   4   

                                 dcj_sydney_lga_rows  
measure                 marker                        
first_quartile_price    -                         15  
median_price            -                         15  
third_quartile_price    -                         15  
mean_price              -                         15  
sales_count             s                        133  
                        -                         15  
quarterly_median_change -                         15  
                        <blank>                    0  
annual_median_change    -                         15  
                        <blank>                    0  
quarterly_count_change  -                         15  
                        <blank>                    0  
annual_count_change     -                         15  
                        <blank>                    0

## 7. Small samples

For DCJ Sydney LGAs, summarise published numeric sales counts by dwelling type and tally the two count markers. Apply the strict fewer-than-30 threshold to numeric counts. A dash is known to be below 30. An `s` could represent exactly 30, so the workbook cannot give an exact total below 30. Check for absent LGA-quarter-type rows separately.

In [7]:
type_rows = sydney_lgas.loc[sydney_lgas["dwelling_type"].isin(["Non Strata", "Strata"])].copy()
type_rows["numeric_sales_count"] = as_number(type_rows["sales_count"])
print("Distribution of numeric sales counts per LGA and quarter")
display(type_rows.groupby("dwelling_type")["numeric_sales_count"].describe().round(1))

sample_summary = type_rows.groupby("dwelling_type").size().rename("published_rows").to_frame()
sample_summary["numeric_below_30"] = (
    type_rows["numeric_sales_count"].lt(30).groupby(type_rows["dwelling_type"]).sum()
)
for marker, column in [("-", "dash_10_or_fewer"), ("s", "small_sample_up_to_30")]:
    sample_summary[column] = (
        type_rows["sales_count"].eq(marker).groupby(type_rows["dwelling_type"]).sum()
    )
sample_summary["minimum_below_30"] = (
    sample_summary["numeric_below_30"] + sample_summary["dash_10_or_fewer"]
)
sample_summary["maximum_below_30"] = (
    sample_summary["minimum_below_30"]
    + sample_summary["small_sample_up_to_30"]
)
display(sample_summary.T)

key_columns = ["quarter_date", "lga", "dwelling_type"]
expected_keys = pd.MultiIndex.from_product(
    [
        layout["quarter_date"],
        sorted(sydney_membership["lga"].unique()),
        ["Non Strata", "Strata"],
    ],
    names=key_columns,
)
present_keys = pd.MultiIndex.from_frame(type_rows[key_columns])
duplicate_keys = present_keys.duplicated().sum()
print("Duplicate Sydney LGA-quarter-type keys", duplicate_keys)
print("Expected Sydney LGA-quarter-type keys absent from the workbooks")
missing_keys = expected_keys.difference(present_keys)
display(missing_keys.to_frame(index=False))

Distribution of numeric sales counts per LGA and quarter


,count,mean,std,min,25%,50%,75%,max
dwelling_type,,,,,,,,
Non Strata,949.0,319.1,265.5,31.0,126.0,264.0,429.0,1683.0
Strata,877.0,315.6,231.9,31.0,155.0,261.0,418.0,1354.0


dwelling_type,Non Strata,Strata
published_rows,986,985
numeric_below_30,0,0
dash_10_or_fewer,0,15
small_sample_up_to_30,37,93
minimum_below_30,0,15
maximum_below_30,37,108


Duplicate Sydney LGA-quarter-type keys 0
Expected Sydney LGA-quarter-type keys absent from the workbooks


,quarter_date,lga,dwelling_type
0,2019-04-01,Wollondilly,Strata


## 8. Spot-check candidates

Locate two median prices and one sales count in the latest workbook. Use the Greater Sydney ring rows. Median values are in thousands of dollars and the count is in sales.

In [8]:
checks = [
    ("Blacktown", "Non Strata", "median_price"),
    ("Sydney", "Strata", "median_price"),
    ("Parramatta", "Strata", "sales_count"),
]
latest_sydney = sydney_lgas.loc[sydney_lgas["filename"].eq(representative)]
spot_checks = []
for lga, dwelling_type, measure in checks:
    matching_rows = (
        latest_sydney["lga"].eq(lga)
        & latest_sydney["dwelling_type"].eq(dwelling_type)
    )
    row = latest_sydney.loc[matching_rows].iloc[0]
    column_letter = excel_columns[columns.index(measure)]
    spot_checks.append({
        "lga": lga,
        "quarter_date": row["quarter_date"],
        "dwelling_type": dwelling_type,
        "measure": measure,
        "raw_value": row[measure],
        "cell": f"{column_letter}{row['excel_row']}",
    })
print(f"Workbook {representative}\nWorksheet LGA")
display(pd.DataFrame(spot_checks).set_index("lga"))

Workbook sales-tables-march-2026-quarter.xlsx
Worksheet LGA


,quarter_date,dwelling_type,measure,raw_value,cell
lga,,,,,
Blacktown,2026-01-01,Non Strata,median_price,1250,G1865
Sydney,2026-01-01,Strata,median_price,967,G1815
Parramatta,2026-01-01,Strata,sales_count,835,J1851


## Part B — Candidate factors

Compare candidate sources with the 34 LGA labels already published in DCJ's Sydney rings. This is a coverage check for T02. Kang's D04 uses these 34 LGAs. The factor choices are still open. The 2019 price-level candidate uses the sales data in Part A. Individual property sales were screened using source documentation.

Use the published ABS codes to inspect name differences and data coverage. The list of alternate labels below identifies the source rows for this check. No file of renamed labels is produced.

In [9]:
import json

dcj_names = set(sydney_lgas["lga"].unique())
preview_lgas = {"Blacktown", "Sydney", "Wollondilly"}
preview_codes = {"10750", "17200", "18400"}

boundary_file = raw_dir / "abs_asgs/LGA_2025_Sydney_region.geojson"
with boundary_file.open(encoding="utf-8") as geo_file:
    boundary_features = json.load(geo_file)["features"]

abs_label_variants = {
    "Bayside (NSW)", "Campbelltown (NSW)", "Central Coast (NSW)",
    "Ku-ring-gai", "Sutherland", "The Hills",
}
region_labels = dcj_names | abs_label_variants
sydney_codes = {
    feature["properties"]["lga_code_2025"] for feature in boundary_features
    if feature["properties"]["lga_name_2025"] in region_labels
}
assert len(sydney_codes) == len(dcj_names) == 34

def show_label_coverage(source_labels):
    labels = set(source_labels.dropna())
    matched = dcj_names & labels
    print(f"Exact labels ({len(matched)}/34)", ", ".join(sorted(matched)) or "none")
    print("DCJ labels without an exact match", ", ".join(sorted(dcj_names - matched)) or "none")
    print("Other source labels in this scope", ", ".join(sorted(labels - dcj_names)) or "none")

### C2. Median weekly rent

Read Table 1 on the `LGA` sheet of the four 2019 rent issues. These are DCJ's NSW reporting LGAs and rings. The workbooks do not name a single dated ABS LGA edition. Inspect the `House` and `Flat/Unit` rows with `Total` bedrooms. Column H gives median weekly rent in dollars for newly lodged bonds, and column J gives the new bond count.

Issue 129 prints an additional June caption in A1. The September title in F1, the cover and the archive identify the intended quarter. The rent categories describe physical dwelling type. A future gross yield would use these as proxies for DCJ's title-based sale categories. Annualising rent and dividing by price is deferred. [DCJ archive](https://dcj.nsw.gov.au/about-us/families-and-communities-statistics/housing-rent-and-sales/previous-rent-and-sales-reports.html)

In [10]:
rent_files = sorted((raw_dir / "dcj_rent").glob("*.xlsx"))
rent_tables = []
for path in rent_files:
    workbook = load_workbook(path, read_only=True, data_only=True)
    try:
        sheet = workbook["LGA"]
        rows = list(sheet.values)
    finally:
        workbook.close()
    header = next(index for index, row in enumerate(rows) if row[4] == "Dwelling Types")
    print(path.name, "LGA", "header row", header + 1)
    print("Published table rows", sum(row[4] is not None for row in rows[header + 1:]))

    table = pd.DataFrame(
        [(i, *row[:6], row[7], row[9]) for i, row in
         enumerate(rows[header + 1:], start=header + 2) if row[4] is not None],
        columns=[
            "excel_row", "gmr", "greater_sydney", "ring", "lga", "dwelling_type",
            "bedrooms", "median_weekly_rent", "new_bonds",
        ],
        dtype=object,
    )
    selected = table.loc[
        table["gmr"].eq(gmr_name)
        & table["greater_sydney"].eq("Greater Sydney")
        & table["ring"].isin(rings)
        & table["lga"].ne("Total")
        & table["bedrooms"].eq("Total")
        & table["dwelling_type"].isin(["House", "Flat/Unit"])
    ].copy()
    print("Sydney rows", len(selected), "distinct LGAs", selected["lga"].nunique())
    print("Blank or suppressed median rents", as_number(selected["median_weekly_rent"]).isna().sum())
    selected["file"] = path.name
    rent_tables.append(selected)

rent_sydney = pd.concat(rent_tables, ignore_index=True)
show_label_coverage(rent_sydney["lga"])
display(rent_sydney.loc[
    rent_sydney["lga"].isin(preview_lgas),
    ["file", "lga", "dwelling_type", "median_weekly_rent", "new_bonds", "excel_row"],
].sort_values(["lga", "file", "dwelling_type"]))

issue-127-rent-tables-march-2019.xlsx LGA header row 4
Published table rows 33035
Sydney rows 68 distinct LGAs 34
Blank or suppressed median rents 0


issue-128-rent-tables-june-2019.xlsx LGA header row 4
Published table rows 33067
Sydney rows 68 distinct LGAs 34
Blank or suppressed median rents 0


issue-129-rent-tables-september-2019.xlsx LGA header row 4
Published table rows 33028
Sydney rows 68 distinct LGAs 34
Blank or suppressed median rents 0


issue-130-rent-tables-december-2019.xlsx LGA header row 6
Published table rows 33019
Sydney rows 68 distinct LGAs 34
Blank or suppressed median rents 0
Exact labels (34/34) Bayside, Blacktown, Blue Mountains, Burwood, Camden, Campbelltown, Canada Bay, Canterbury-Bankstown, Central Coast, Cumberland, Fairfield, Georges River, Hawkesbury, Hornsby, Hunters Hill, Inner West, Ku-Ring-Gai, Lane Cove, Liverpool, Mosman, North Sydney, Northern Beaches, Parramatta, Penrith, Randwick, Ryde, Strathfield, Sutherland Shire, Sydney, The Hills Shire, Waverley, Willoughby, Wollondilly, Woollahra
DCJ labels without an exact match none
Other source labels in this scope none


,file,lga,dwelling_type,median_weekly_rent,new_bonds,excel_row
41,issue-127-rent-tables-march-2019.xlsx,Blacktown,Flat/Unit,370,456,21650
40,issue-127-rent-tables-march-2019.xlsx,Blacktown,House,470,1840,21629
109,issue-128-rent-tables-june-2019.xlsx,Blacktown,Flat/Unit,370,469,21667
108,issue-128-rent-tables-june-2019.xlsx,Blacktown,House,450,1864,21646
177,issue-129-rent-tables-september-2019.xlsx,Blacktown,Flat/Unit,360,500,21650
176,issue-129-rent-tables-september-2019.xlsx,Blacktown,House,460,2238,21629
245,issue-130-rent-tables-december-2019.xlsx,Blacktown,Flat/Unit,380,596,21654
244,issue-130-rent-tables-december-2019.xlsx,Blacktown,House,460,1844,21633
11,issue-127-rent-tables-march-2019.xlsx,Sydney,Flat/Unit,650,6212,21062
10,issue-127-rent-tables-march-2019.xlsx,Sydney,House,775,573,21041


### C3. Estimated resident population

Read the annual population estimates in `32180DS0004_2001-25.xlsx`, Table 1. The unit is people at 30 June. The latest release recasts the series to ABS LGA 2025 boundaries. The 2016 and 2019 counts would support the baseline-period growth measure. The 2019 and 2025 counts overlap the sales-growth period. Show the published counts only. [ABS Regional population](https://www.abs.gov.au/statistics/people/population/regional-population/2024-25)

In [11]:
erp_file = raw_dir / "abs_erp/32180DS0004_2001-25.xlsx"
erp_years = list(range(2001, 2026))
population = pd.read_excel(
    erp_file, sheet_name="Table 1", header=None, skiprows=6, usecols="A:AA",
    names=["lga_code", "source_label", *erp_years],
)
population = population.loc[pd.to_numeric(population["lga_code"], errors="coerce").notna()]
population_sydney = population.loc[population["lga_code"].astype(str).isin(sydney_codes)]
print(erp_file.name, "Table 1", len(population), "published area rows")
print("Sydney code coverage", population_sydney["lga_code"].nunique(), "of 34")
show_label_coverage(population_sydney["source_label"])
print("Missing Sydney counts across the 25 years", int(population_sydney[erp_years].isna().sum().sum()))
display(population_sydney.loc[
    population_sydney["lga_code"].astype(str).isin(preview_codes),
    ["lga_code", "source_label", 2016, 2019, 2025],
])

32180DS0004_2001-25.xlsx Table 1 548 published area rows
Sydney code coverage 34 of 34
Exact labels (28/34) Blacktown, Blue Mountains, Burwood, Camden, Canada Bay, Canterbury-Bankstown, Cumberland, Fairfield, Georges River, Hawkesbury, Hornsby, Hunters Hill, Inner West, Lane Cove, Liverpool, Mosman, North Sydney, Northern Beaches, Parramatta, Penrith, Randwick, Ryde, Strathfield, Sydney, Waverley, Willoughby, Wollondilly, Woollahra
DCJ labels without an exact match Bayside, Campbelltown, Central Coast, Ku-Ring-Gai, Sutherland Shire, The Hills Shire
Other source labels in this scope Bayside (NSW), Campbelltown (NSW), Central Coast (NSW), Ku-ring-gai, Sutherland, The Hills
Missing Sydney counts across the 25 years 0


,lga_code,source_label,2016,2019,2025
9,10750,Blacktown,348030.0,380186.0,449385.0
105,17200,Sydney,222717.0,229596.0,241797.0
124,18400,Wollondilly,49854.0,52826.0,62080.0


### C4. Median household income

Read G02 from the NSW General Community Profile DataPacks for the 2016 and 2021 Censuses. The field `Median_tot_hhd_inc_weekly` is dollars per week. The CSVs use Census LGA codes. The descriptor workbook supplies each year's original names for the label audit. Inspect 2016 as a pre-period snapshot and 2021 as concurrent.

The 2016 Census preceded the creation of Bayside. Show its two predecessor labels and count their records separately. [ABS Census DataPacks](https://www.abs.gov.au/census/find-census-data/datapacks)

In [12]:
census_dir = raw_dir / "abs_census"
census_files = [
    (2016, "2016Census_geog_desc_1st_and_2nd_release.xlsx", "2016_ASGS_Non-ABS_Structures"),
    (2021, "2021Census_geog_desc_1st_2nd_3rd_release.xlsx", "2021_ASGS_Non_ABS_Structures"),
]
bayside_predecessors = {"11100", "16650"}
income_field = "Median_tot_hhd_inc_weekly"
for year, descriptor_name, sheet_name in census_files:
    code_column = f"LGA_CODE_{year}"
    file = census_dir / f"{year}Census_G02_NSW_LGA.csv"
    income = pd.read_csv(file, dtype={code_column: "string"})
    descriptor = pd.read_excel(census_dir / descriptor_name, sheet_name=sheet_name)
    code = f"Census_Code_{year}"
    name = f"Census_Name_{year}"
    current_codes = {"LGA" + value for value in sydney_codes}
    relevant_codes = set(current_codes)
    if year == 2016:
        relevant_codes.update("LGA" + value for value in bayside_predecessors)
    labels = descriptor.loc[descriptor[code].isin(relevant_codes), [code, name]]
    print(file.name, len(income), "NSW rows")
    print(descriptor_name, sheet_name)
    show_label_coverage(labels[name])
    available = income.loc[income[code_column].isin(current_codes)]
    older = income.loc[income[code_column].isin(relevant_codes - current_codes)]
    print("Current codes represented", available[code_column].nunique(), "of 34")
    print("Predecessor rows present", older[code_column].nunique())
    print("Blank or nonnumeric income in present rows",
          as_number(income.loc[income[code_column].isin(relevant_codes), income_field]).isna().sum())
    display(income.loc[
        income[code_column].isin({"LGA" + value for value in preview_codes}),
        [code_column, income_field],
    ])
    if year == 2016:
        display(labels.loc[labels[code].isin(relevant_codes - current_codes)])

2016Census_G02_NSW_LGA.csv 132 NSW rows
2016Census_geog_desc_1st_and_2nd_release.xlsx 2016_ASGS_Non-ABS_Structures
Exact labels (0/34) none
DCJ labels without an exact match Bayside, Blacktown, Blue Mountains, Burwood, Camden, Campbelltown, Canada Bay, Canterbury-Bankstown, Central Coast, Cumberland, Fairfield, Georges River, Hawkesbury, Hornsby, Hunters Hill, Inner West, Ku-Ring-Gai, Lane Cove, Liverpool, Mosman, North Sydney, Northern Beaches, Parramatta, Penrith, Randwick, Ryde, Strathfield, Sutherland Shire, Sydney, The Hills Shire, Waverley, Willoughby, Wollondilly, Woollahra
Other source labels in this scope Blacktown (C), Blue Mountains (C), Botany Bay (C), Burwood (A), Camden (A), Campbelltown (C) (NSW), Canada Bay (A), Canterbury-Bankstown (A), Central Coast (C) (NSW), Cumberland (A), Fairfield (C), Georges River (A), Hawkesbury (C), Hornsby (A), Hunters Hill (A), Inner West (A), Ku-ring-gai (A), Lane Cove (A), Liverpool (C), Mosman (A), North Sydney (A), Northern Beaches (A),

,LGA_CODE_2016,Median_tot_hhd_inc_weekly
8,LGA10750,1711
105,LGA17200,1926
125,LGA18400,1871


,Census_Code_2016,Census_Name_2016
182,LGA11100,Botany Bay (C)
265,LGA16650,Rockdale (C)


2021Census_G02_NSW_LGA.csv 131 NSW rows
2021Census_geog_desc_1st_2nd_3rd_release.xlsx 2021_ASGS_Non_ABS_Structures
Exact labels (30/34) Blacktown, Blue Mountains, Burwood, Camden, Canada Bay, Canterbury-Bankstown, Cumberland, Fairfield, Georges River, Hawkesbury, Hornsby, Hunters Hill, Inner West, Lane Cove, Liverpool, Mosman, North Sydney, Northern Beaches, Parramatta, Penrith, Randwick, Ryde, Strathfield, Sutherland Shire, Sydney, The Hills Shire, Waverley, Willoughby, Wollondilly, Woollahra
DCJ labels without an exact match Bayside, Campbelltown, Central Coast, Ku-Ring-Gai
Other source labels in this scope Bayside (NSW), Campbelltown (NSW), Central Coast (NSW), Ku-ring-gai
Current codes represented 34 of 34
Predecessor rows present 0
Blank or nonnumeric income in present rows 0


,LGA_CODE_2021,Median_tot_hhd_inc_weekly
9,LGA10750,2107
105,LGA17200,2212
124,LGA18400,2151


### C5. Distance to Sydney CBD

Use the official ABS LGA 2025 boundaries from its Regional Population feature service. An envelope limits the number of downloaded features. The service returns full intersecting polygons, with no clipping or simplification requested. The ABS state and date-specific labels remain as published. The Sydney General Post Office point comes from the NSW State Heritage Register, item 5045424, listing 00763.

DuckDB spatial computes each polygon's area-weighted geometric centroid in Australian Albers (EPSG:9473). Transform it back to longitude and latitude, then measure the straight-line distance along the WGS84 ellipsoid. Use separate geometry for each LGA and include all polygon parts. Display Blacktown, Sydney and the four large outer LGAs. [ABS boundary layer](https://geo.abs.gov.au/arcgis/rest/services/Hosted/LGA_Regional_Population_2025/FeatureServer/1), [NSW State Heritage Register layer](https://mapprod3.environment.nsw.gov.au/arcgis/rest/services/HMS/Heritage/MapServer/5)

In [13]:
import duckdb

gpo_file = raw_dir / "abs_asgs/Sydney_GPO_00763.geojson"
with gpo_file.open(encoding="utf-8") as geo_file:
    gpo_feature = json.load(geo_file)["features"][0]
gpo_lon, gpo_lat = gpo_feature["geometry"]["coordinates"]
print(gpo_file.name, "1 point")
print("GPO latitude", gpo_lat)
print("GPO longitude", gpo_lon)

with duckdb.connect() as spatial:
    spatial.execute("INSTALL spatial")
    spatial.execute("LOAD spatial")
    spatial.execute("SET geometry_always_xy = true")
    polygons = spatial.table_function("ST_Read", [str(boundary_file)])
    print(boundary_file.name, len(boundary_features), "polygon features", polygons.columns)
    projected = polygons.project(
        "lga_code_2025, lga_name_2025, "
        "ST_Transform(geom, 'EPSG:4326', 'EPSG:9473', always_xy := true) AS area_geom"
    )
    centres = projected.project("lga_code_2025, lga_name_2025, ST_Centroid(area_geom) AS centre")
    lonlat = centres.project(
        "lga_code_2025, lga_name_2025, "
        "ST_Transform(centre, 'EPSG:9473', 'EPSG:4326', always_xy := true) AS lonlat"
    )
    # Keep longitude first for both GeoJSON coordinates and distance.
    distances = lonlat.project(
        "lga_code_2025, lga_name_2025, ST_Y(lonlat) AS centroid_latitude, "
        "ST_X(lonlat) AS centroid_longitude, "
        f"ST_Distance_Spheroid(lonlat, "
        f"ST_Point({gpo_lon}, {gpo_lat})) / 1000 AS distance_km"
    ).df()

sydney_distances = distances.loc[distances["lga_code_2025"].isin(sydney_codes)]
show_label_coverage(sydney_distances["lga_name_2025"])
print("Missing Sydney centroids or distances",
      sydney_distances[["centroid_latitude", "centroid_longitude", "distance_km"]].isna().any(axis=1).sum())
coordinate_examples = preview_lgas | {"Central Coast (NSW)", "Blue Mountains", "Hawkesbury"}
examples = sydney_distances.loc[sydney_distances["lga_name_2025"].isin(coordinate_examples)]
examples = examples.set_index("lga_name_2025").rename(columns={
    "lga_code_2025": "code", "centroid_latitude": "latitude",
    "centroid_longitude": "longitude",
})
display(examples.round({"latitude": 6, "longitude": 6, "distance_km": 2}))

Sydney_GPO_00763.geojson 1 point
GPO latitude -33.86775531912647
GPO longitude 151.20770702900006


LGA_2025_Sydney_region.geojson 43 polygon features ['OGC_FID', 'lga_code_2025', 'lga_name_2025', 'geom']


Exact labels (28/34) Blacktown, Blue Mountains, Burwood, Camden, Canada Bay, Canterbury-Bankstown, Cumberland, Fairfield, Georges River, Hawkesbury, Hornsby, Hunters Hill, Inner West, Lane Cove, Liverpool, Mosman, North Sydney, Northern Beaches, Parramatta, Penrith, Randwick, Ryde, Strathfield, Sydney, Waverley, Willoughby, Wollondilly, Woollahra
DCJ labels without an exact match Bayside, Campbelltown, Central Coast, Ku-Ring-Gai, Sutherland Shire, The Hills Shire
Other source labels in this scope Bayside (NSW), Campbelltown (NSW), Central Coast (NSW), Ku-ring-gai, Sutherland, The Hills
Missing Sydney centroids or distances 0


,code,latitude,longitude,distance_km
lga_name_2025,,,,
Blacktown,10750,-33.740914,150.864086,34.79
Blue Mountains,10900,-33.697431,150.411147,76.16
Central Coast (NSW),11650,-33.307863,151.285535,62.52
Hawkesbury,13800,-33.334510,150.773405,71.57
Sydney,17200,-33.889440,151.202763,2.45
Wollondilly,18400,-34.101865,150.443696,75.22


The geometric centroid weights land area. This can give large tracts of bushland or rural land more influence than the places residents cluster. A population-weighted centroid is feasible using the ABS population grid or small-area Census counts aligned to the LGA boundary vintage. That alternative is a method choice for later. [ABS population grid downloads](https://www.abs.gov.au/statistics/people/population/regional-population/2024-25)

Compare the 2016 to 2021 and annual 2021 to 2025 population-weighted correspondence tables. Inspect positive weights to other codes and changes in names at the same code. The weights describe population allocation between statistical areas. Consult the ABS change notes for changes that do not move population. [ABS correspondences](https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-3-july-2021-june-2026/access-and-downloads/correspondences), [ABS LGA change history](https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-3-july-2021-june-2026/non-abs-structures/local-government-areas)

In [14]:
correspondence_files = sorted((raw_dir / "abs_asgs").glob("CG_*.csv"))
for path in correspondence_files:
    correspondence = pd.read_csv(path, dtype="string")
    old_code, old_name, new_code, new_name, weight = correspondence.columns[:5]
    scoped = correspondence.loc[
        correspondence[old_code].isin(sydney_codes | bayside_predecessors)
        | correspondence[new_code].isin(sydney_codes)
    ]
    same_code = scoped[old_code].fillna("").eq(scoped[new_code].fillna(""))
    positive = pd.to_numeric(scoped[weight], errors="coerce").gt(0)
    code_moves = scoped.loc[~same_code & positive, [old_code, old_name, new_code, new_name, weight]]
    label_changes = scoped.loc[same_code & scoped[old_name].ne(scoped[new_name]), [old_name, new_name]]
    print(path.name, len(scoped), "relevant rows")
    print("Positive weights to a different code or unallocated", len(code_moves))
    print("Changed labels at the same code", len(label_changes))
    if not code_moves.empty:
        display(code_moves)
    if not label_changes.empty and "2016" not in path.name:
        display(label_changes)

CG_2022_LGA_2023_LGA.csv 34 relevant rows
Positive weights to a different code or unallocated 0
Changed labels at the same code 0
CG_LGA_2016_LGA_2021.csv 40 relevant rows
Positive weights to a different code or unallocated 7
Changed labels at the same code 33


,LGA_CODE_2016,LGA_NAME_2016,LGA_CODE_2021,LGA_NAME_2021,RATIO_FROM_TO
10,10750,Blacktown (C),12380,Cumberland,1.80E-05
15,11100,Botany Bay (C),10500,Bayside (NSW),1
23,11450,Camden (A),11500,Campbelltown (NSW),5.92E-05
24,11500,Campbelltown (C) (NSW),11450,Camden,0.0004083
40,12380,Cumberland (A),16260,Parramatta,1.14E-05
102,16650,Rockdale (C),10500,Bayside (NSW),1
111,17150,Sutherland Shire (A),<NA>,<NA>,2.27E-05


CG_LGA_2020_LGA_2021.csv 39 relevant rows
Positive weights to a different code or unallocated 5
Changed labels at the same code 34


,LGA_CODE_2020,LGA_NAME_2020,LGA_CODE_2021,LGA_NAME_2021,RATIO_FROM_TO
10,10750,Blacktown (C),12380,Cumberland,1.80E-05
22,11450,Camden (A),11500,Campbelltown (NSW),5.92E-05
23,11500,Campbelltown (C) (NSW),11450,Camden,0.0004083
39,12380,Cumberland (A),16260,Parramatta,1.14E-05
109,17150,Sutherland Shire (A),<NA>,<NA>,2.27E-05


,LGA_NAME_2020,LGA_NAME_2021
5,Bayside (A),Bayside (NSW)
9,Blacktown (C),Blacktown
13,Blue Mountains (C),Blue Mountains
18,Burwood (A),Burwood
21,Camden (A),Camden
24,Campbelltown (C) (NSW),Campbelltown (NSW)
25,Canada Bay (A),Canada Bay
26,Canterbury-Bankstown (A),Canterbury-Bankstown
28,Central Coast (C) (NSW),Central Coast (NSW)
38,Cumberland (A),Cumberland


CG_LGA_2021_LGA_2022.csv 34 relevant rows
Positive weights to a different code or unallocated 0
Changed labels at the same code 2


,LGA_NAME_2021,LGA_NAME_2022
104,Sutherland Shire,Sutherland
109,The Hills Shire,The Hills


CG_LGA_2023_LGA_2024.csv 34 relevant rows
Positive weights to a different code or unallocated 0
Changed labels at the same code 0
CG_LGA_2024_LGA_2025.csv 34 relevant rows
Positive weights to a different code or unallocated 0
Changed labels at the same code 0


### C6. New dwelling supply proxy

Read the ABS Building Approvals count of total approved dwellings for years ending 30 June. The June 2017 NSW datacubes supply 2015–16 and 2016–17. The two ABS Data by region workbooks supply numeric approval counts for the later years. Their printed year ranges include earlier rows marked with dashes. The oldest cube retains pre-amalgamation councils, the second uses the 2016-era LGA labels, and the latest Data by region workbook uses LGA 2021. Consult the source editions and label audit before comparing periods.

An approval is an indication of planned supply. The proposed scale denominator is existing private dwellings from the 2016 Census G32, including occupied and unoccupied private dwellings. This section shows approvals and stock separately, with no rate or series combined. Years within the outcome window are concurrent. The 2019 financial year includes months on both sides of the start of the baseline. [ABS Building Approvals](https://www.abs.gov.au/AUSSTATS/abs@.nsf/DetailsPage/8731.0Jun%202017), [ABS Data by region, 2015–20](https://www.abs.gov.au/methodologies/data-region-methodology/2015-20), [ABS Data by region, 2011–25](https://www.abs.gov.au/methodologies/data-region-methodology/2011-25)

In [15]:
import xlrd

legacy_approval_files = [
    (2016, "87310do004_201706.xls"),
    (2017, "87310do005_201706.xls"),
]
for year, filename in legacy_approval_files:
    legacy_book = xlrd.open_workbook(raw_dir / "supply" / filename, on_demand=True)
    try:
        sheet = legacy_book.sheet_by_name("Table_1")
        rows = []
        for row_number in range(6, sheet.nrows):
            row = sheet.row_values(row_number)
            rows.append([row[0], row[1], row[4]])
    finally:
        legacy_book.release_resources()
    approvals = pd.DataFrame(rows, columns=["Code", "Label", "Total dwellings"])
    approvals = approvals.loc[approvals["Code"].astype(str).str.fullmatch(r"\d{5}", na=False)]
    direct = approvals.loc[approvals["Code"].astype(str).isin(sydney_codes)]
    predecessors = approvals.loc[approvals["Code"].astype(str).isin(bayside_predecessors)]
    print(filename, "Table_1", "FY ended", year, "NSW LGA rows", len(approvals))
    print("Current codes represented", len(direct), "of 34")
    print("Current codes absent", sorted(sydney_codes - set(direct["Code"].astype(str))))
    show_label_coverage(pd.concat([direct["Label"], predecessors["Label"]]))
    print("Blank or suppressed counts among present current codes", as_number(direct["Total dwellings"]).isna().sum())
    display(direct.loc[direct["Code"].astype(str).isin(preview_codes)])

regional_approval_files = [
    (raw_dir / "_archives/14100DO0003_2015-20.xlsx", "A:C,BL", [2018, 2019, 2020]),
    (raw_dir / "supply/14100DO0003_2011-25.xlsx", "A:C,AJ", [2021, 2022, 2023, 2024, 2025]),
]
approval_field = "Total dwelling units (no.)"
for path, columns_to_read, useful_years in regional_approval_files:
    approvals = pd.read_excel(path, sheet_name="Table 2", header=6, usecols=columns_to_read)
    approvals = approvals.loc[pd.to_numeric(approvals["Code"], errors="coerce").notna()]
    codes = pd.to_numeric(approvals["Code"], errors="coerce").astype("Int64").astype(str)
    subset = approvals.loc[codes.isin(sydney_codes)]
    print(path.name, "Table 2", "published Australian LGA-year rows", len(approvals))
    print("Current codes represented", subset["Code"].nunique(), "of 34")
    show_label_coverage(subset["Label"])
    print("Blank or nonnumeric approvals by FY-end year among the Sydney codes")
    missing = as_number(subset[approval_field]).isna()
    display(missing.groupby(subset["Year"]).sum().rename("missing").to_frame())
    display(approvals.loc[codes.isin(preview_codes) & approvals["Year"].isin(useful_years)])

stock_file = census_dir / "2016Census_G32_NSW_LGA.csv"
stock_code = "LGA_CODE_2016"
stock_field = "Total_PDs_Dwellings"
stock = pd.read_csv(stock_file, dtype={stock_code: "string"})
stock_sydney = stock.loc[stock[stock_code].isin({"LGA" + value for value in sydney_codes})]
print(stock_file.name, len(stock), "NSW rows", "present current codes", len(stock_sydney), "of 34")
print("Blank or nonnumeric stock among present rows", as_number(stock_sydney[stock_field]).isna().sum())
display(stock_sydney.loc[
    stock_sydney[stock_code].isin({"LGA" + value for value in preview_codes}),
    [stock_code, stock_field],
])

87310do004_201706.xls Table_1 FY ended 2016 NSW LGA rows 154
Current codes represented 26 of 34
Current codes absent ['10500', '11570', '11650', '12380', '12930', '14170', '15990', '16260']
Exact labels (0/34) none
DCJ labels without an exact match Bayside, Blacktown, Blue Mountains, Burwood, Camden, Campbelltown, Canada Bay, Canterbury-Bankstown, Central Coast, Cumberland, Fairfield, Georges River, Hawkesbury, Hornsby, Hunters Hill, Inner West, Ku-Ring-Gai, Lane Cove, Liverpool, Mosman, North Sydney, Northern Beaches, Parramatta, Penrith, Randwick, Ryde, Strathfield, Sutherland Shire, Sydney, The Hills Shire, Waverley, Willoughby, Wollondilly, Woollahra
Other source labels in this scope Blacktown (C), Blue Mountains (C), Botany Bay (C), Burwood (A), Camden (A), Campbelltown (C), Canada Bay (A), Fairfield (C), Hawkesbury (C), Hornsby (A), Hunters Hill (A), Ku-ring-gai (A), Lane Cove (A), Liverpool (C), Mosman (A), North Sydney (A), Penrith (C), Randwick (C), Rockdale (C), Ryde (C), Str

,Code,Label,Total dwellings
12,10750,Blacktown (C),4273.0
122,17200,Sydney (C),5740.0
147,18400,Wollondilly (A),476.0


87310do005_201706.xls Table_1 FY ended 2017 NSW LGA rows 132
Current codes represented 33 of 34
Current codes absent ['10500']
Exact labels (0/34) none
DCJ labels without an exact match Bayside, Blacktown, Blue Mountains, Burwood, Camden, Campbelltown, Canada Bay, Canterbury-Bankstown, Central Coast, Cumberland, Fairfield, Georges River, Hawkesbury, Hornsby, Hunters Hill, Inner West, Ku-Ring-Gai, Lane Cove, Liverpool, Mosman, North Sydney, Northern Beaches, Parramatta, Penrith, Randwick, Ryde, Strathfield, Sutherland Shire, Sydney, The Hills Shire, Waverley, Willoughby, Wollondilly, Woollahra
Other source labels in this scope Blacktown (C), Blue Mountains (C), Botany Bay (C), Burwood (A), Camden (A), Campbelltown (C) (NSW), Canada Bay (A), Canterbury-Bankstown (A), Central Coast (C) (NSW), Cumberland (A), Fairfield (C), Georges River (A), Hawkesbury (C), Hornsby (A), Hunters Hill (A), Inner West (A), Ku-ring-gai (A), Lane Cove (A), Liverpool (C), Mosman (A), North Sydney (A), Northern 

,Code,Label,Total dwellings
9,10750,Blacktown (C),4290.0
106,17200,Sydney (C),4897.0
126,18400,Wollondilly (A),388.0


14100DO0003_2015-20.xlsx Table 2 published Australian LGA-year rows 3808
Current codes represented 34 of 34
Exact labels (0/34) none
DCJ labels without an exact match Bayside, Blacktown, Blue Mountains, Burwood, Camden, Campbelltown, Canada Bay, Canterbury-Bankstown, Central Coast, Cumberland, Fairfield, Georges River, Hawkesbury, Hornsby, Hunters Hill, Inner West, Ku-Ring-Gai, Lane Cove, Liverpool, Mosman, North Sydney, Northern Beaches, Parramatta, Penrith, Randwick, Ryde, Strathfield, Sutherland Shire, Sydney, The Hills Shire, Waverley, Willoughby, Wollondilly, Woollahra
Other source labels in this scope Bayside (A), Blacktown (C), Blue Mountains (C), Burwood (A), Camden (A), Campbelltown (C) (NSW), Canada Bay (A), Canterbury-Bankstown (A), Central Coast (C) (NSW), Cumberland (A), Fairfield (C), Georges River (A), Hawkesbury (C), Hornsby (A), Hunters Hill (A), Inner West (A), Ku-ring-gai (A), Lane Cove (A), Liverpool (C), Mosman (A), North Sydney (A), Northern Beaches (A), Parramatt

,missing
Year,
2011.0,34
2015.0,34
2016.0,34
2017.0,34
2018.0,0
2019.0,0
2020.0,0


... (2 outputs left out)

,missing
Year,
2011.0,34
2016.0,34
2017.0,34
2018.0,34
2019.0,34
2020.0,34
2021.0,0
2022.0,0
2023.0,0


,Code,Label,Year,Total dwelling units (no.)
105,10750,Blacktown,2021.0,6730
106,10750,Blacktown,2022.0,5716
107,10750,Blacktown,2023.0,4288
108,10750,Blacktown,2024.0,2391
109,10750,Blacktown,2025.0,2488
1161,17200,Sydney,2021.0,2232
1162,17200,Sydney,2022.0,1184
1163,17200,Sydney,2023.0,1108
1164,17200,Sydney,2024.0,690
1165,17200,Sydney,2025.0,1751


2016Census_G32_NSW_LGA.csv 132 NSW rows present current codes 33 of 34
Blank or nonnumeric stock among present rows 0


,LGA_CODE_2016,Total_PDs_Dwellings
8,LGA10750,106205
105,LGA17200,96925
125,LGA18400,16048


### C7. Sydney All groups CPI

Read the quarterly Sydney index from ABS Table 17. The records are capital-city index numbers. One Sydney series applies across the LGAs and provides a possible real-price deflator. Show three observations and the date coverage. [ABS Consumer Price Index](https://www.abs.gov.au/statistics/economy/price-indexes-and-inflation/consumer-price-index-australia/sep-2026)

In [16]:
from datetime import datetime

cpi_file = raw_dir / "abs_cpi/6401017.xlsx"
cpi_raw = pd.read_excel(cpi_file, sheet_name="Data1", header=None, usecols="A:B")
cpi_observations = cpi_raw.loc[cpi_raw[0].map(lambda value: isinstance(value, datetime))]
print(cpi_file.name, "Data1", len(cpi_observations), "quarterly observations")
print(cpi_raw.iloc[0, 1], "series", cpi_raw.iloc[9, 1], "unit", cpi_raw.iloc[1, 1])
print("First and last dates", cpi_observations[0].min(), cpi_observations[0].max())
print("Blank Sydney index values", cpi_observations[1].isna().sum())
print("LGA coverage: one Sydney series shared by all 34; no LGA rows or labels.")
display(cpi_observations.loc[cpi_observations[0].isin([
    datetime(2019, 3, 1), datetime(2026, 3, 1), datetime(2026, 9, 1),
])].set_axis(["source_date", "sydney_all_groups_index"], axis=1))

6401017.xlsx Data1 313 quarterly observations
Index Numbers ;  All groups CPI ;  Sydney ; series A2325806K unit Index Numbers
First and last dates 1948-09-01 00:00:00 2026-09-01 00:00:00
Blank Sydney index values 0
LGA coverage: one Sydney series shared by all 34; no LGA rows or labels.


,source_date,sydney_all_groups_index
292,2019-03-01 00:00:00,79.97
320,2026-03-01 00:00:00,101.81
322,2026-09-01 00:00:00,103.71


### C8. RBA cash rate target

Read RBA Statistical Table A2. Column C is the new target in per cent and column B is the change in percentage points. The event date is in column A. A national policy series can mark changes on a timeline across all LGAs. Show three published target changes. [RBA cash rate target and A2](https://www.rba.gov.au/statistics/cash-rate/)

In [17]:
rba_file = raw_dir / "rba/a02hist.xlsx"
rba_raw = pd.read_excel(rba_file, sheet_name="Data", header=None, usecols="A:C")
rba_events = rba_raw.loc[rba_raw[0].map(lambda value: isinstance(value, datetime))]
print(rba_file.name, "Data", len(rba_events), "dated target observations")
print("Target field", rba_raw.iloc[1, 2], "series", rba_raw.iloc[10, 2], "unit", rba_raw.iloc[5, 2])
print("First and last event dates", rba_events[0].min(), rba_events[0].max())
print("Missing target among event rows", rba_events[2].isna().sum())
print("LGA coverage: one national series shared by all 34; no LGA rows or labels.")
display(rba_events.loc[rba_events[0].isin([
    datetime(2019, 6, 5), datetime(2026, 3, 18), datetime(2026, 11, 4),
])].set_axis(["event_date", "change_pp", "target_pct"], axis=1))

a02hist.xlsx Data 100 dated target observations
Target field New Cash Rate Target series ARBAMPCNCRT unit Per cent
First and last event dates 1990-01-23 00:00:00 2026-11-04 00:00:00
Missing target among event rows 0
LGA coverage: one national series shared by all 34; no LGA rows or labels.


,event_date,change_pp,target_pct
84,2019-06-05 00:00:00,-0.25,1.25
107,2026-03-18 00:00:00,0.25,4.1
110,2026-11-04 00:00:00,0.25,4.85


Sources for the Census DataPacks, Regional Population, Building Approvals, Data by region, correspondences and CPI are the Australian Bureau of Statistics. ABS data used with permission from the Australian Bureau of Statistics. Distance calculations are based on ABS boundaries and the Heritage NSW point. Rent tables are © State of New South Wales (Department of Communities and Justice). The source for A2 is Reserve Bank of Australia, 2026. [ABS DataPacks](https://www.abs.gov.au/census/find-census-data/datapacks), [DCJ copyright](https://dcj.nsw.gov.au/statements/copyright-and-disclaimer.html), [RBA copyright](https://www.rba.gov.au/copyright/index.html)